In [1]:
import pandas as pd

from homework.homework import (
    load_data,
    clean_data,
    split,
    build_pipeline,
    tune_hyperparameters,
    compute_metrics,
    compute_confusion,
    save_model,
    save_metrics,
)

In [2]:
df_train = clean_data(load_data("files/input/train_data.csv.zip"))
df_test = clean_data(load_data("files/input/test_data.csv.zip"))

X_train, y_train = split(df_train)
X_test, y_test = split(df_test)

df_train.head()


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,310000,1,3,1,32,0,0,0,0,0,...,84373,57779,14163,8295,6000,4000,3000,1000,2000,0
1,10000,2,3,1,49,-1,-1,-2,-1,2,...,1690,1138,930,0,0,2828,0,182,0,1
2,50000,1,2,1,28,-1,-1,-1,0,-1,...,45975,1300,43987,0,46257,2200,1300,43987,1386,0
3,80000,2,3,1,52,2,2,3,3,3,...,40748,39816,40607,3700,1600,1600,0,1600,1600,1
4,270000,1,1,2,34,1,2,0,0,2,...,22448,15490,17343,0,4000,2000,0,2000,2000,0


In [3]:
pipeline = build_pipeline()

grid = tune_hyperparameters(
    pipeline=pipeline,
    x_train=X_train,
    y_train=y_train,
    scoring="balanced_accuracy",
)

print("Mejor balanced accuracy (CV):", grid.best_score_)
print("Mejores hiperparámetros:", grid.best_params_)


Fitting 10 folds for each of 1 candidates, totalling 10 fits
[CV] END classifier__alpha=0.26, classifier__hidden_layer_sizes=(50, 30, 40, 60), classifier__learning_rate_init=0.001, feature_selection__k=20, pca__n_components=None; total time=  24.0s
[CV] END classifier__alpha=0.26, classifier__hidden_layer_sizes=(50, 30, 40, 60), classifier__learning_rate_init=0.001, feature_selection__k=20, pca__n_components=None; total time=  41.2s
[CV] END classifier__alpha=0.26, classifier__hidden_layer_sizes=(50, 30, 40, 60), classifier__learning_rate_init=0.001, feature_selection__k=20, pca__n_components=None; total time=  47.0s
[CV] END classifier__alpha=0.26, classifier__hidden_layer_sizes=(50, 30, 40, 60), classifier__learning_rate_init=0.001, feature_selection__k=20, pca__n_components=None; total time= 1.0min
[CV] END classifier__alpha=0.26, classifier__hidden_layer_sizes=(50, 30, 40, 60), classifier__learning_rate_init=0.001, feature_selection__k=20, pca__n_components=None; total time= 1.1min

In [4]:
y_pred_train = grid.predict(X_train)
y_pred_test = grid.predict(X_test)


In [5]:
metrics_train = compute_metrics("train", y_train, y_pred_train)
metrics_test = compute_metrics("test", y_test, y_pred_test)

cm_train = compute_confusion("train", y_train, y_pred_train)
cm_test = compute_confusion("test", y_test, y_pred_test)

metrics_train, metrics_test, cm_train, cm_test


({'type': 'metrics',
  'dataset': 'train',
  'precision': 0.7025,
  'balanced_accuracy': 0.663,
  'recall': 0.3719,
  'f1_score': 0.4863},
 {'type': 'metrics',
  'dataset': 'test',
  'precision': 0.6846,
  'balanced_accuracy': 0.6696,
  'recall': 0.3872,
  'f1_score': 0.4946},
 {'type': 'cm_matrix',
  'dataset': 'train',
  'true_0': {'predicted_0': 15484, 'predicted_1': 744},
  'true_1': {'predicted_0': 2968, 'predicted_1': 1757}},
 {'type': 'cm_matrix',
  'dataset': 'test',
  'true_0': {'predicted_0': 6733, 'predicted_1': 340},
  'true_1': {'predicted_0': 1168, 'predicted_1': 738}})

In [7]:
save_model(grid, "files/models/model.pkl.gz")

results = [metrics_train, metrics_test, cm_train, cm_test]
save_metrics(results, "files/output/metrics.json")
